# M9 신경망으로 가는 다리 — 실습 (W14)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. **퍼셉트론 손계산**(AND·OR)과 **손 가중치 MLP의 XOR 완주**(y=0,1,1,0)를 코드로 검산한다 ⭐
2. XOR 400점에서 **로지스틱 0.5(필연의 실패) vs MLP 1.0**을 실측하고 결정경계를 비교한다
3. **은닉 뉴런 스윕 + 시드 12개**로 "존재 ≠ 학습"(은닉 2는 3/12만 성공)을 확인한다

**7단계 멘탈모델 초점:** 모델의 한계 → 층을 쌓는다(표현을 바꾼다)

## Part A. 퍼셉트론 손계산 검산 — AND·OR ⭐
가중치 w=(1,1) 고정, 편향 b만 바꿔 봅니다(판정: z>0이면 1). AND의 b를 종이에서 먼저 정한 뒤 검산하세요.

In [ ]:
import numpy as np                                     # 수치 계산

pts = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])       # 네 입력 (x1, x2)
w1, w2 = 1, 1                                          # 가중치는 (1, 1)로 고정

z_and = w1*pts[:, 0] + w2*pts[:, 1] + (___)            # ✍️ 빈칸: AND의 편향 b — 경계 직선 x1+x2=1.5가 되는 그 수
print('AND z:', z_and, '-> 판정:', (z_and > 0).astype(int))  # z=-1.5/-0.5/-0.5/0.5 → 0,0,0,1

z_or = w1*pts[:, 0] + w2*pts[:, 1] + (-0.5)            # OR의 편향 b=-0.5 (경계 x1+x2=0.5)
print('OR  z:', z_or, '-> 판정:', (z_or > 0).astype(int))    # 0,1,1,1

> **검산 포인트:** AND는 z = −1.5/−0.5/−0.5/**0.5** → 판정 0,0,0,**1** ✓ / OR는 0,**1,1,1** ✓ — 직선 하나를 **어디에** 긋느냐(=b)의 문제. 그런데 XOR(0,1,1,0)은 **4줄 모순 증명**(reading §5)으로 어떤 w, b로도 불가능 — 다음 Part에서 층을 쌓아 넘습니다.

## Part B. 세트피스 — 손 가중치 MLP로 XOR 완주 ⭐
은닉 뉴런 2개를 끼웁니다: h₁=ReLU(x₁+x₂), h₂=ReLU(x₁+x₂−1), y=h₁−2h₂. ReLU는 "0 미만은 0" 문지기(`np.maximum`). 표를 종이에서 먼저 채우고 검산하세요.

In [ ]:
x1, x2 = pts[:, 0], pts[:, 1]                          # 네 점의 좌표
z1 = x1 + x2                                           # 은닉 1의 가중합
h1 = np.maximum(___, z1)                               # ✍️ 빈칸: ReLU — 이 값 미만을 잘라냄(문지기의 기준)
z2 = x1 + x2 - 1                                       # 은닉 2의 가중합
h2 = np.maximum(0, z2)                                 # ReLU
y_hat = h1 - ___ * h2                                  # ✍️ 빈칸: 출력층 — h2를 몇 배로 빼나?
print('h1:', h1)                                       # [0 1 1 2]
print('h2:', h2)                                       # [0 0 0 1] — "둘 다 1인가?" 탐지기
print('y :', y_hat, '(정답 XOR: 0 1 1 0)')             # 완주!
print('새 좌표 (h1,h2):', list(zip(h1.tolist(), h2.tolist())))  # (0,0) (1,0) (1,0) (2,1)

> **검산 포인트:** y = **[0 1 1 0]** — XOR 완주(2학기 D1에서 이 표를 텐서로 재현). 새 좌표 (h₁,h₂) = (0,0), (1,0), (1,0), (2,1) — 클래스 1 두 점이 **같은 점으로 겹치고** 전체가 **직선 하나로 나뉘는 배치**. "은닉층 = 표현을 바꾼다"의 실체입니다(M7의 x² 접어 올리기와 같은 정신 — 단, 신경망은 접는 법을 **학습**).

## Part C. XOR 데이터 400점 만들기
네 모서리에 잡음 낀 군집을 만들고, 대각선끼리 같은 클래스로 둡니다(XOR).

In [ ]:
import matplotlib.pyplot as plt                        # 그래프
from matplotlib.colors import ListedColormap           # 색 지정

rng = np.random.RandomState(0)                         # 재현용 난수
n = 100                                                # 모서리당 점 수
X = np.r_[rng.randn(n, 2)*0.15 + [0, 0],               # (0,0) 근처
          rng.randn(n, 2)*0.15 + [1, 1],               # (1,1) 근처
          rng.randn(n, 2)*0.15 + [0, 1],               # (0,1) 근처
          rng.randn(n, 2)*0.15 + [1, 0]]               # (1,0) 근처
y = np.r_[np.zeros(n), np.zeros(n),                    # (0,0),(1,1) → 0
          np.ones(n), np.ones(n)].astype(int)          # (0,1),(1,0) → 1  (= XOR)

plt.scatter(X[:, 0], X[:, 1], c=y,                     # XOR 데이터 산점도
            cmap=ListedColormap(['#2563eb', '#ef4444']), s=12)
plt.xlabel('x1'); plt.ylabel('x2')                     # 축(영어)
plt.title('XOR data (400 points)'); plt.show()         # 제목(영어)

> 파랑(클래스 0)은 한 대각선, 빨강(클래스 1)은 반대 대각선. reading §5의 4줄 모순 증명이 그대로 적용되는 배치입니다 — 직선 하나로 가를 수 있을까요?

## Part D. 로지스틱(선형) 시도 → 필연의 0.5

In [ ]:
from sklearn.linear_model import LogisticRegression    # 선형 분류기

lin = LogisticRegression()                             # 로지스틱(직선 경계 — M5)
lin.fit(X, ___)                                        # ✍️ 빈칸: 학습 정답 라벨
print('선형 모델 정확도:', round(lin.score(X, y), 3))   # 0.5 — 동전 던지기와 정확히 같음

> **0.5** — 4줄 증명이 보장하는 건 **만점 불가**까지입니다(최선의 직선은 한 모서리를 고립시켜 네 군집 중 셋 ≈ **0.75**까지 가능). 그런데 로지스틱은 정확도가 아니라 **로그 손실을 최소화**(M4) — 이 대칭 배치에선 어느 방향의 직선도 손실 이득이 없어 가중치가 사실상 0(무정보)이 되고, 그 결과가 0.5입니다.

## Part E. MLP(은닉층) → 1.0

In [ ]:
from sklearn.neural_network import MLPClassifier       # 다층 퍼셉트론(은닉층 있음)

mlp = MLPClassifier(hidden_layer_sizes=(8, ___),       # ✍️ 빈칸: 은닉층 2개(각 8뉴런) — 두 번째 층의 뉴런 수
                    max_iter=2000, random_state=0)     # 충분히 학습
mlp.fit(X, y)                                          # 학습
print('MLP 정확도:', round(mlp.score(X, y), 3))         # 1.0 — XOR 해결!

## Part F. 결정경계 비교 — 직선 vs 곡선

In [ ]:
xx, yy = np.meshgrid(np.linspace(-1, 2, 300),          # 평면 격자
                     np.linspace(-1, 2, 300))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))        # 나란히 두 그림
for ax, mdl, name in zip(axes, [lin, mlp], ['Linear (fail)', 'MLP (solve)']):
    Z = mdl.___(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)  # ✍️ 빈칸: 격자 전체를 판정하는 메서드
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#93c5fd', '#fca5a5']))
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=ListedColormap(['#2563eb', '#ef4444']), s=10)
    ax.set_title(name)                                 # 제목(영어)
plt.tight_layout(); plt.show()                         # 표시

> 왼쪽(선형)은 **직선** 경계라 어디에 그어도 두 군집이 섞이고, 오른쪽(MLP)은 **곡선** 경계로 네 군집을 정확히 나눕니다. Part B에서 손으로 만든 "표현 바꾸기"를, 여기서는 학습이 해냈습니다.

## Part G. 은닉 뉴런의 여유 — 존재 ≠ 학습
은닉 뉴런 수를 바꾸고(시드 고정), 은닉 2는 시드 12개로 반복해 "운"을 셉니다.

In [ ]:
import warnings                                        # 수렴 경고 숨김(실패 시드에서 발생)
warnings.filterwarnings('ignore')

for hs in [(2,), (4,), (___,)]:                        # ✍️ 빈칸: 안정 성공을 볼 은닉 수(본문의 그 수)
    m = MLPClassifier(hidden_layer_sizes=hs, max_iter=2000, random_state=0).fit(X, y)
    print(f'hidden {hs}: {round(m.score(X, y), 3)}')   # 0.985 / 0.983 / 1.0 (시드 0)

wins = 0                                               # 은닉 2의 "운" 실험
for seed in range(12):                                 # 시드 12개
    m = MLPClassifier(hidden_layer_sizes=(2,), max_iter=2000, random_state=seed).fit(X, y)
    wins += m.score(X, y) >= 0.98                      # 성공(≈1.0)이면 카운트
print('은닉 2: 12시드 중 성공', wins, '개')             # 3개 — 풀 가중치는 존재(Part B!)해도 학습은 운

> **은닉 2 = 12시드 중 3개만 성공** — Part B의 손 가중치가 "은닉 2로 푸는 가중치는 존재한다"의 증거인데도요. **"존재한다" ≠ "찾을 수 있다"** — 뉴런 여유(4는 10/12, 8은 12/12)가 학습의 운을 이깁니다. 2학기 D1에서 같은 현상을 PyTorch로 다시 만납니다.

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "XOR 불가능의 4줄 모순 증명을 내가 써 볼 테니 빈틈을 찾아 줘."
- "손 가중치 표에서 h₂가 '둘 다 1 탐지기'인 이유를 설명해 볼게 — 허점을 찔러 줘."
- "은닉 2가 12시드 중 3개만 성공하는 이유를 '존재 vs 학습'으로 설명해 볼게."
- "NAND(둘 다 1일 때만 0)를 푸는 w, b를 손으로 정해 볼 테니 채점해 줘."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 퍼셉트론 손계산(AND·OR)을 검산하고, **XOR 불가능(4줄 모순)**을 확인했다 — 실측 로지스틱 **0.5**
2. **손 가중치 MLP로 XOR을 완주**했다(y=0,1,1,0) — 새 좌표 (0,0)(1,0)(1,0)(2,1) = "표현을 바꾼다"
3. MLP 실측 **1.0** + 은닉 스윕(2는 3/12, 4는 10/12)으로 **"존재 ≠ 학습"**을 봤다

**스스로 점검**
- [ ] AND·OR의 b를 스스로 정할 수 있다
- [ ] 4줄 모순 증명을 재현할 수 있다
- [ ] 손 가중치 표의 한 행을 종이에서 계산할 수 있다
- [ ] 새 좌표가 왜 "표현 바꾸기"의 증거인지 말할 수 있다
- [ ] 은닉 2의 3/12이 무엇과 무엇의 구분인지 안다

**🔹심화 (선택)**
- NAND·NOR를 푸는 w, b를 손으로 정하고 Part A 방식으로 검산해 보세요.
- Part B의 출력에 계단(y_hat > 0.5)을 씌워 0/1 판정으로 바꿔 보세요.
- Part G에서 `max_iter`를 200으로 줄이면 성공 수가 어떻게 변하는지 확인해 보세요.

**1학기 끝!** 다음 학기(2학기 D1): 오늘의 XOR 표를 **텐서**로 재현하고, 가중치를 경사하강으로 **학습**시킵니다.